# E3 - YOLO26s + MultiSEAM trên DataSet2

Notebook này thực hiện ablation độc lập E3: chỉ thêm MultiSEAM vào các fusion point P3/8 và P4/16 được phát hiện từ runtime của YOLO26s.

- Giữ nguyên dataset, class order, train/val split và protocol của baseline.
- Không thêm P2, iAFF, SEAM riêng, DySample, SCConv, CBAM, ECA, AFPN, ADown, custom loss hoặc class weighting.
- Không dùng test set để chọn model.
- Output riêng: `helmet_yolo26s_multiseam_dataset2`; không ghi đè baseline/E1/E2.
- Kết luận KEEP/REJECT chỉ được tạo sau khi có metric thực tế.

In [ ]:
# Colab dependencies (chạy một lần trong runtime mới).
%pip install -q ultralytics==8.4.155
%pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
# Thiết lập dataset trên Google Drive, giữ nguyên dataset của baseline.
from google.colab import drive
import os
import shutil
from pathlib import Path
import gdown

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/DetectedHelmets/DataSet2', exist_ok=True)
os.makedirs('/content/DetectedHelmets/Code', exist_ok=True)
os.chdir('/content/DetectedHelmets')

archive_path = Path('/content/DetectedHelmets/dataset.rar')
data_root = Path('/content/DetectedHelmets/DataSet2')
drive_dataset_path = Path('/content/drive/MyDrive/DataSet2')
local_dataset_path = data_root

if drive_dataset_path.exists():
    if local_dataset_path.is_symlink():
        local_dataset_path.unlink()
    elif local_dataset_path.exists() and local_dataset_path.resolve() != drive_dataset_path.resolve():
        shutil.rmtree(local_dataset_path)
    if not local_dataset_path.exists():
        local_dataset_path.symlink_to(drive_dataset_path, target_is_directory=True)
    print(f'Dataset source: {local_dataset_path} -> {drive_dataset_path}')
else:
    if archive_path.exists():
        archive_path.unlink()
    file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', str(archive_path), quiet=False, fuzzy=True)
    os.system(f'unrar x -o+ {archive_path} {data_root}/')
    print(f'Dataset extracted to: {data_root}')

## 1. Environment, dataset và protocol baseline

In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import ultralytics
import yaml
from PIL import Image
from ultralytics import YOLO

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
assert torch.cuda.is_available(), 'GPU chưa được bật; protocol yêu cầu device=0.'

In [ ]:
DATA_YAML_PATH = Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
MODEL_NAME = 'yolo26s.pt'
ULTRALYTICS_VERSION = '8.4.155'
assert ultralytics.__version__ == ULTRALYTICS_VERSION, (ultralytics.__version__, ULTRALYTICS_VERSION)
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_multiseam_dataset2'
BASELINE_EXPERIMENT_NAME = 'helmet_yolo26s_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
BASELINE_BEST_MODEL_PATH = OUTPUT_DIR / BASELINE_EXPERIMENT_NAME / 'weights' / 'best.pt'
VALIDATION_SPLIT = 'val'

TRAIN_CONFIG = {
    'epochs': 50,
    'imgsz': 640,
    'batch': 16,
    'device': 0,
    'workers': 4,
    'optimizer': 'auto',
    'seed': 0,
    'deterministic': True,
    'patience': 15,
    'pretrained': True,
}

with DATA_YAML_PATH.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

raw_names = data_config.get('names', [])
if isinstance(raw_names, dict):
    raw_names = [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))]
CLASS_NAMES = list(raw_names)
EXPECTED_CLASS_NAMES = ['motorcycle', 'rider_with_helmet', 'rider_without_helmet']

assert DATA_YAML_PATH.exists(), f'Không tìm thấy data.yaml: {DATA_YAML_PATH}'
assert data_config.get('nc') == 3
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, (CLASS_NAMES, EXPECTED_CLASS_NAMES)
assert EXPERIMENT_NAME != BASELINE_EXPERIMENT_NAME
print(f'Dataset: {DATA_YAML_PATH}')
print(f'Classes: {CLASS_NAMES}')
print(f'Output: {EXPERIMENT_DIR}')
print(f'Baseline checkpoint: {BASELINE_BEST_MODEL_PATH}')
print(f'Train protocol: {TRAIN_CONFIG}')

## 2. Inspect kiến trúc YOLO26s thực tế trong runtime

Không đoán layer index. Cell dưới đây in architecture và hook các `Concat` để đo spatial stride của từng fusion point. P3/8 và P4/16 được chọn theo kết quả probe; P5/32 và số detection scale được giữ nguyên.

In [ ]:
base_model = YOLO(MODEL_NAME)
assert getattr(base_model, 'task', None) == 'detect', base_model.task
print('--- YOLO26s runtime architecture ---')
print(base_model.model)
print('--- YOLO26s model.info(imgsz=640) ---')
base_model.info(verbose=True, imgsz=640)

def probe_concat_points(yolo_model, imgsz=640):
    records = []
    handles = []

    def make_hook(index, module):
        def hook(_module, inputs, output):
            source = inputs[0] if len(inputs) == 1 else inputs
            tensors = list(source) if isinstance(source, (list, tuple)) else [source]
            tensor_shapes = [tuple(t.shape) for t in tensors if torch.is_tensor(t)]
            if not tensor_shapes:
                return
            output_shape = tuple(output.shape) if torch.is_tensor(output) else None
            records.append({
                'index': index,
                'from': getattr(module, 'f', None),
                'input_shapes': tensor_shapes,
                'output_shape': output_shape,
                'height': tensor_shapes[0][-2],
                'width': tensor_shapes[0][-1],
                'stride': imgsz // tensor_shapes[0][-1],
            })
        return hook

    for index, module in enumerate(yolo_model.model.model):
        if module.__class__.__name__ == 'Concat':
            handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training:
        yolo_model.model.train()
    for handle in handles:
        handle.remove()
    return sorted(records, key=lambda item: item['index'])

fusion_probe = probe_concat_points(base_model, imgsz=640)
print('--- Runtime neck fusion points ---')
for record in fusion_probe:
    print(record)

detect_indices = [
    index for index, module in enumerate(base_model.model.model)
    if module.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}
]
assert detect_indices, 'Không tìm thấy Detect head trong YOLO26s runtime graph.'
detect_index = min(detect_indices)
DETECT_HEAD_COUNT = len(detect_indices)
neck_candidates = [record for record in fusion_probe if record['index'] < detect_index and record['stride'] in (8, 16)]
assert neck_candidates, 'Không tìm thấy fusion P3/P4 trước Detect head.'

# Chọn đúng một fusion gần Detect nhất cho mỗi feature level. Không thay mọi Concat cùng stride.
selected_by_stride = {}
for stride in (8, 16):
    candidates = [record for record in neck_candidates if record['stride'] == stride]
    assert candidates, f'Không tìm thấy fusion output P{stride // 8 + 2}/{stride} trước Detect.'
    selected_by_stride[stride] = max(candidates, key=lambda record: record['index'])

fusion_specs = [
    {
        'level': f'P{stride // 8 + 2}/{stride}',
        'index': selected_by_stride[stride]['index'],
        'from': selected_by_stride[stride]['from'],
        'input_channels': [shape[1] for shape in selected_by_stride[stride]['input_shapes']],
        'spatial': (selected_by_stride[stride]['height'], selected_by_stride[stride]['width']),
        'stride': stride,
    }
    for stride in (8, 16)
]
MULTISEAM_EXPECTED_COUNT = 2
assert len(fusion_specs) == MULTISEAM_EXPECTED_COUNT
assert {spec['stride'] for spec in fusion_specs} == {8, 16}
print(f'Detect head runtime index: {detect_index}')
print('--- Exactly two MultiSEAM insertion points (runtime-discovered) ---')
for spec in fusion_specs:
    print(spec)

## 3. MultiSEAM: kiến trúc và lý do chọn vị trí

MultiSEAM dùng đúng cấu trúc tham chiếu: ba nhánh DcovN với patch_size lần lượt 3, 5 và 7; mỗi DcovN bắt đầu bằng Conv2d(c1,c2,kernel_size=patch_size,stride=patch_size), SiLU, BatchNorm2d, tiếp theo là các Residual depthwise blocks rồi Conv2d 1x1, SiLU, BatchNorm2d. Sau mỗi nhánh là AdaptiveAvgPool2d(1), cộng với nhánh AvgPool(x), lấy trung bình, qua FC C -> C//16 -> C với ReLU/Sigmoid, rồi torch.exp. Output là x * y, giữ nguyên B,C,H,W.

P3/8 và P4/16 được ưu tiên vì rider/head/helmet có thể nhỏ hoặc bị che khuất. Concat gốc vẫn được giữ trong wrapper `ConcatMultiSEAM`: Concat theo channel trước, sau đó mới chạy MultiSEAM. Không concat các feature maps y0/y1/y2 theo spatial và không thêm nhánh detection/P2.

In [ ]:
class Residual(nn.Module):
    def __init__(self, module):
        super().__init__()
        self.module = module

    def forward(self, x):
        return x + self.module(x)


class DcovN(nn.Module):
    """Reference DcovN branch: patch projection, residual depthwise blocks, projection."""

    def __init__(self, c1, c2, depth, kernel_size=3, patch_size=3):
        super().__init__()
        self.patch_projection = nn.Sequential(
            nn.Conv2d(c1, c2, kernel_size=patch_size, stride=patch_size),
            nn.SiLU(),
            nn.BatchNorm2d(c2),
        )
        residual_block = lambda: Residual(nn.Sequential(
            nn.Conv2d(c2, c2, kernel_size=kernel_size, stride=1, padding=kernel_size // 2, groups=c2),
            nn.SiLU(),
            nn.BatchNorm2d(c2),
        ))
        self.residual_blocks = nn.Sequential(*(residual_block() for _ in range(depth)))
        self.output_projection = nn.Sequential(
            nn.Conv2d(c2, c2, kernel_size=1),
            nn.SiLU(),
            nn.BatchNorm2d(c2),
        )

    def forward(self, x):
        x = self.patch_projection(x)
        x = self.residual_blocks(x)
        return self.output_projection(x)


class MultiSEAM(nn.Module):
    """Shape-preserving reference MultiSEAM with three DcovN branches."""

    def __init__(self, c1, c2=None, depth=1):
        super().__init__()
        self.c1 = int(c1)
        self.c2 = int(c2 if c2 is not None else c1)
        if self.c1 != self.c2:
            raise ValueError('MultiSEAM must preserve channels, so c1 must equal c2.')
        self.DcovN0 = DcovN(self.c1, self.c2, depth=depth, kernel_size=3, patch_size=3)
        self.DcovN1 = DcovN(self.c1, self.c2, depth=depth, kernel_size=3, patch_size=5)
        self.DcovN2 = DcovN(self.c1, self.c2, depth=depth, kernel_size=3, patch_size=7)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(self.c2 // 16, 1)
        self.fc = nn.Sequential(
            nn.Linear(self.c2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, self.c2),
            nn.Sigmoid(),
        )

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.c1:
            raise ValueError(f'MultiSEAM expected BCHW with C={self.c1}, got {tuple(x.shape)}')
        y0 = self.avg_pool(self.DcovN0(x))
        y1 = self.avg_pool(self.DcovN1(x))
        y2 = self.avg_pool(self.DcovN2(x))
        y3 = self.avg_pool(x)
        y = (y0 + y1 + y2 + y3) / 4.0
        y = self.fc(y.flatten(1)).view(x.shape[0], self.c2, 1, 1)
        y = torch.exp(y)
        return x * y


class ConcatMultiSEAM(nn.Module):
    """The original runtime Concat module followed by exactly one MultiSEAM."""

    def __init__(self, concat_module, input_channels):
        super().__init__()
        self.concat = concat_module
        self.input_channels = tuple(int(value) for value in input_channels)
        self.output_channels = sum(self.input_channels)
        self.multi_seam = MultiSEAM(self.output_channels, self.output_channels, depth=1)

    def forward(self, inputs):
        if not isinstance(inputs, (list, tuple)) or len(inputs) != len(self.input_channels):
            raise ValueError('ConcatMultiSEAM expects the original fusion input list.')
        if len({tuple(item.shape[-2:]) for item in inputs}) != 1:
            raise ValueError('ConcatMultiSEAM inputs must have the same spatial shape.')
        fused = self.concat(inputs)
        if fused.shape[1] != self.output_channels:
            raise ValueError(f'Original Concat changed channels: expected {self.output_channels}, got {fused.shape[1]}')
        return self.multi_seam(fused)

In [ ]:
def count_multiseam(module):
    return sum(1 for submodule in module.modules() if submodule.__class__.__name__ == 'MultiSEAM')

def sum_tensors(value):
    if torch.is_tensor(value):
        return value.float().sum()
    if isinstance(value, (list, tuple)):
        terms = [sum_tensors(item) for item in value]
        return sum(terms) if terms else torch.zeros(())
    if isinstance(value, dict):
        terms = [sum_tensors(item) for item in value.values()]
        return sum(terms) if terms else torch.zeros(())
    return torch.zeros(())

# Focused unit checks before integration.
unit_module = MultiSEAM(40, 40, depth=1)
unit_input = torch.randn(2, 40, 32, 32, requires_grad=True)
unit_output = unit_module(unit_input)
assert unit_output.shape == unit_input.shape
unit_output.mean().backward()
assert any(parameter.grad is not None for parameter in unit_module.parameters())
class UnitConcat(nn.Module):
    def forward(self, inputs):
        return torch.cat(inputs, dim=1)

unit_wrapper = ConcatMultiSEAM(UnitConcat(), [16, 24])
wrapper_output = unit_wrapper([torch.randn(2, 16, 32, 32), torch.randn(2, 24, 32, 32)])
assert wrapper_output.shape == (2, 40, 32, 32)
print('MultiSEAM unit checks: DcovN branches, shape preservation and backward gradient OK')

In [ ]:
def copy_graph_metadata(replacement, old_module, fallback_index, fallback_from):
    # Ultralytics graph runner and model.info use these layer attributes.
    replacement.i = getattr(old_module, 'i', fallback_index)
    replacement.f = getattr(old_module, 'f', fallback_from)
    replacement.type = getattr(old_module, 'type', replacement.__class__.__name__)
    replacement.np = sum(parameter.numel() for parameter in replacement.parameters())
    return replacement

def replace_runtime_fusions(yolo_model, specs):
    for spec in specs:
        old_module = yolo_model.model.model[spec['index']]
        assert old_module.__class__.__name__ == 'Concat', (spec, old_module.__class__.__name__)
        replacement = ConcatMultiSEAM(old_module, spec['input_channels'])
        replacement = copy_graph_metadata(replacement, old_module, spec['index'], spec['from'])
        yolo_model.model.model[spec['index']] = replacement
    return yolo_model

def build_custom_multiseam_model(model_name=MODEL_NAME):
    return replace_runtime_fusions(YOLO(model_name), fusion_specs)

custom_model = build_custom_multiseam_model()
source_state = base_model.model.state_dict()
target_state = custom_model.model.state_dict()
compatible_state = {
    key: value
    for key, value in source_state.items()
    if key in target_state and tuple(value.shape) == tuple(target_state[key].shape)
}
load_report = custom_model.model.load_state_dict(compatible_state, strict=False)

source_parameters = dict(base_model.model.named_parameters())
target_parameters = dict(custom_model.model.named_parameters())
compatible_parameter_keys = {
    key for key in compatible_state if key in source_parameters and key in target_parameters
}
transferred_parameter_count = sum(source_parameters[key].numel() for key in compatible_parameter_keys)
new_parameter_count = sum(value.numel() for key, value in target_parameters.items() if key not in compatible_parameter_keys)

model = custom_model
print(f'Pretrained parameter values transferred: {transferred_parameter_count:,}')
print(f'New MultiSEAM parameter values: {new_parameter_count:,}')
print(f'Compatible state tensors transferred: {len(compatible_state):,}')
print(f'New/unmatched state tensors: {len(load_report.missing_keys):,}')
print(f'Unexpected state keys: {load_report.unexpected_keys}')
print(f'MultiSEAM modules in graph: {count_multiseam(model.model)}')
assert count_multiseam(model.model) == MULTISEAM_EXPECTED_COUNT == 2
assert count_multiseam(model.model) == 2

## 4. Preflight graph: interface, forward và backward

In [ ]:
# Verify exactly two replacements preserve the original fusion output channel count and spatial size.
assert count_multiseam(model.model) == MULTISEAM_EXPECTED_COUNT == 2
for spec in fusion_specs:
    module = model.model.model[spec['index']]
    height, width = spec['spatial']
    inputs = [torch.randn(2, channels, height, width) for channels in spec['input_channels']]
    output = module(inputs)
    assert output.shape == (2, sum(spec['input_channels']), height, width), (spec, output.shape)

device = next(model.model.parameters()).device
model.model.eval()
with torch.no_grad():
    dummy_output = model.model(torch.randn(2, 3, 640, 640, device=device))
print('Dummy forward torch.randn(2, 3, 640, 640): OK')

model.model.train()
model.model.zero_grad(set_to_none=True)
graph_input = torch.randn(2, 3, 640, 640, device=device, requires_grad=True)
graph_output = model.model(graph_input)
graph_loss = sum_tensors(graph_output)
assert graph_loss.requires_grad, 'Dummy output không nằm trong computational graph.'
graph_loss.backward()
multiseam_gradients = [
    parameter.grad
    for submodule in model.model.modules()
    if submodule.__class__.__name__ == 'MultiSEAM'
    for parameter in submodule.parameters()
    if parameter.grad is not None
]
assert multiseam_gradients and all(torch.isfinite(gradient).all() for gradient in multiseam_gradients)
print(f'Computational graph check: OK ({len(multiseam_gradients)} MultiSEAM parameter gradients)')
model.model.eval()

## 5. Train E3 với custom DetectionTrainer

`model.train(...)` không được gọi trực tiếp vì Ultralytics có thể rebuild graph từ YAML. Trainer dưới đây luôn trả về đúng `custom_pytorch_model`, sau đó kiểm tra module count trước khi epoch đầu tiên chạy.

In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer

custom_pytorch_model = model.model
multiseam_before_train = count_multiseam(custom_pytorch_model)
print('MultiSEAM BEFORE TRAIN:', multiseam_before_train)
assert multiseam_before_train == MULTISEAM_EXPECTED_COUNT == 2
assert count_multiseam(custom_pytorch_model) == 2

class CustomMultiSEAMTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        # Do not rebuild from cfg/YAML; keep the already modified PyTorch graph.
        return custom_pytorch_model

    def setup_model(self):
        # Guard all setup paths against replacing the custom graph.
        self.model = custom_pytorch_model
        return self.model

TRAIN_OVERRIDES = {
    'model': MODEL_NAME,
    'data': str(DATA_YAML_PATH),
    **TRAIN_CONFIG,
    'project': str(OUTPUT_DIR),
    'name': EXPERIMENT_NAME,
    'save': True,
    'save_period': 10,
    'plots': True,
    'exist_ok': False,
}

trainer = CustomMultiSEAMTrainer(overrides=TRAIN_OVERRIDES, _callbacks=model.callbacks)
trainer.model = custom_pytorch_model
trainer_multiseam_count = count_multiseam(trainer.model)
print('MultiSEAM INSIDE TRAINER:', trainer_multiseam_count)
assert trainer_multiseam_count == MULTISEAM_EXPECTED_COUNT == 2
assert count_multiseam(trainer.model) == 2
print(trainer.model)

# The first epoch starts only after the mandatory in-trainer assertion above.
train_results = trainer.train()

EXPERIMENT_VERSION_FILE = EXPERIMENT_DIR / 'ultralytics_version.txt'
EXPERIMENT_VERSION_FILE.write_text(
    f'ultralytics=={ultralytics.__version__}\n'
    f'model={MODEL_NAME}\n'
    f'multiseam_expected_count={MULTISEAM_EXPECTED_COUNT}\n',
    encoding='utf-8',
)
print(f'Experiment version recorded at: {EXPERIMENT_VERSION_FILE}')

## 6. Load `best.pt` mà không làm mất MultiSEAM

Ưu tiên load trực tiếp. Nếu Ultralytics không deserialize được custom module, notebook rebuilds lại architecture từ `fusion_specs` rồi load state dict từ checkpoint. Cả hai đường đều bắt buộc giữ `MultiSEAM`.

In [ ]:
def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def extract_checkpoint_state(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict):
        for key in ('model', 'ema', 'state_dict'):
            value = payload.get(key)
            if isinstance(value, nn.Module):
                return value.state_dict()
            if isinstance(value, dict):
                tensor_state = {name: item for name, item in value.items() if torch.is_tensor(item)}
                if tensor_state:
                    return tensor_state
        tensor_state = {name: item for name, item in payload.items() if torch.is_tensor(item)}
        if tensor_state:
            return tensor_state
    raise TypeError(f'Không tìm thấy state dict trong checkpoint type={type(payload)}')

def load_custom_multiseam_checkpoint(path):
    direct_error = None
    try:
        direct_model = YOLO(str(path))
        if count_multiseam(direct_model.model) > 0:
            return direct_model, 'direct'
        direct_error = RuntimeError('Direct load returned a graph without MultiSEAM.')
    except Exception as exc:
        direct_error = exc
    print(f'Direct checkpoint load unavailable: {direct_error}')

    rebuilt = build_custom_multiseam_model()
    payload = torch_load_checkpoint(str(path))
    checkpoint_state = extract_checkpoint_state(payload)
    if checkpoint_state and all(name.startswith('module.') for name in checkpoint_state):
        checkpoint_state = {name[len('module.'):]: value for name, value in checkpoint_state.items()}
    target_state = rebuilt.model.state_dict()
    compatible = {
        name: value
        for name, value in checkpoint_state.items()
        if name in target_state and tuple(value.shape) == tuple(target_state[name].shape)
    }
    if not compatible:
        raise RuntimeError('Checkpoint không có tensor nào tương thích với architecture MultiSEAM đã rebuild.')
    report = rebuilt.model.load_state_dict(compatible, strict=False)
    print(f'Rebuilt checkpoint load: {len(compatible):,} compatible tensors')
    print(f'Rebuilt missing keys: {len(report.missing_keys):,}')
    assert count_multiseam(rebuilt.model) == MULTISEAM_EXPECTED_COUNT == 2
    return rebuilt, 'rebuilt'

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy best.pt: {BEST_MODEL_PATH}')
best_model, checkpoint_load_mode = load_custom_multiseam_checkpoint(BEST_MODEL_PATH)
multiseam_count_best = count_multiseam(best_model.model)
print('MultiSEAM IN BEST MODEL:', multiseam_count_best)
assert multiseam_count_best == MULTISEAM_EXPECTED_COUNT == 2
assert count_multiseam(best_model.model) == 2
print(f'Checkpoint load mode: {checkpoint_load_mode}')

## 7. Validation metrics trên cùng validation set

E3 chỉ gọi `split='val'`. Không evaluate test set và không dùng test images để chọn model.

In [ ]:
def safe_f1(precision_value, recall_value):
    denominator = precision_value + recall_value
    return 2.0 * precision_value * recall_value / denominator if denominator else 0.0

def extract_metrics(split_metrics, split_name):
    summary = {
        'Precision': float(split_metrics.box.mp),
        'Recall': float(split_metrics.box.mr),
        'mAP50': float(split_metrics.box.map50),
        'mAP50-95': float(split_metrics.box.map),
    }
    summary['F1'] = safe_f1(summary['Precision'], summary['Recall'])
    summary['by_class'] = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision_value, recall_value, ap50_value, ap5095_value = split_metrics.box.class_result(class_id)
        class_summary = {
            'Precision': float(precision_value),
            'Recall': float(recall_value),
            'AP50': float(ap50_value),
            'AP50-95': float(ap5095_value),
        }
        class_summary['F1'] = safe_f1(class_summary['Precision'], class_summary['Recall'])
        summary['by_class'][class_name] = class_summary
    print(f'--- {split_name} overall ---')
    for metric_name in ('Precision', 'Recall', 'F1', 'mAP50', 'mAP50-95'):
        print(f'{metric_name:10s}: {summary[metric_name]:.4f}')
    print('--- Per-class metrics ---')
    for class_name, values in summary['by_class'].items():
        values_text = ' | '.join(f'{key}={values[key]:.4f}' for key in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95'))
        print(f'{class_name:24s} | {values_text}')
    return summary

metrics = best_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=True)
validation_summary = extract_metrics(metrics, 'E3 validation')
no_helmet_metrics = validation_summary['by_class']['rider_without_helmet']
print('--- rider_without_helmet ---')
for metric_name in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95'):
    print(f'{metric_name:10s}: {no_helmet_metrics[metric_name]:.4f}')

# Explicitly keep test evaluation disabled for this ablation.
test_metrics = None
test_summary = None

## 8. Results, confusion matrix và error analysis

In [ ]:
PLOT_SPECS = [
    ('results.png', 'Training results'),
    ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Normalized confusion matrix'),
    ('BoxPR_curve.png', 'Box PR curve'),
    ('BoxF1_curve.png', 'Box F1 curve'),
    ('BoxP_curve.png', 'Box Precision curve'),
    ('BoxR_curve.png', 'Box Recall curve'),
]

def find_experiment_plot(filename):
    candidates = list(EXPERIMENT_DIR.rglob(filename)) if EXPERIMENT_DIR.exists() else []
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

for filename, title in PLOT_SPECS:
    plot_path = find_experiment_plot(filename)
    if plot_path is None:
        print(f'Không tìm thấy {filename} trong {EXPERIMENT_DIR}.')
        continue
    plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
    plt.imshow(Image.open(plot_path))
    plt.axis('off')
    plt.title(f'{title} - {plot_path.parent.name}')
    plt.show()

def get_confusion_matrix_array(split_metrics):
    confusion_object = getattr(split_metrics, 'confusion_matrix', None)
    matrix = getattr(confusion_object, 'matrix', None)
    if matrix is None:
        return None
    matrix = np.asarray(matrix)
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    return matrix if matrix.shape == expected_shape else None

confusion_matrix_array = get_confusion_matrix_array(metrics)
if confusion_matrix_array is None:
    print('Không đọc được confusion matrix object trực tiếp; không tự tạo số liệu lỗi.')
else:
    # Ultralytics detection confusion matrix: rows=predicted, columns=actual.
    confusion_labels = CLASS_NAMES + ['background']
    confusion_table = pd.DataFrame(confusion_matrix_array, index=[f'predicted:{label}' for label in confusion_labels], columns=[f'actual:{label}' for label in confusion_labels])
    display(confusion_table)
    label_to_index = {label: index for index, label in enumerate(confusion_labels)}

    def error_count(actual_label, predicted_label):
        return float(confusion_matrix_array[label_to_index[predicted_label], label_to_index[actual_label]])

    error_rows = [
        {'Error': 'rider_without_helmet -> rider_with_helmet', 'Count': error_count('rider_without_helmet', 'rider_with_helmet')},
        {'Error': 'rider_without_helmet -> background', 'Count': error_count('rider_without_helmet', 'background')},
        {'Error': 'background -> rider_without_helmet', 'Count': error_count('background', 'rider_without_helmet')},
    ]
    error_analysis = pd.DataFrame(error_rows)
    display(error_analysis)

## 9. Complexity và inference speed trên cùng validation images

In [ ]:
valid_img_dir = Path('/content/DetectedHelmets/DataSet2/DataSet2/valid/images')
valid_images = sorted([str(path) for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG') for path in valid_img_dir.glob(pattern)])
if not valid_images:
    raise FileNotFoundError(f'Không tìm thấy validation images: {valid_img_dir}')

def extract_gflops(info):
    if isinstance(info, (tuple, list)):
        for value in reversed(info):
            try:
                return float(value)
            except (TypeError, ValueError):
                continue
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                try:
                    return float(info[key])
                except (TypeError, ValueError):
                    pass
    return None

def model_gflops(yolo_model):
    return extract_gflops(yolo_model.info(verbose=True, imgsz=640))

def benchmark_inference(yolo_model, image_paths):
    for image_path in image_paths[:2]:
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
    elapsed = []
    for image_path in image_paths:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed.append(time.perf_counter() - start_time)
    milliseconds = float(np.mean(elapsed) * 1000.0)
    return milliseconds, 1000.0 / milliseconds if milliseconds else 0.0

speed_images = valid_images[:min(50, len(valid_images))]
if not BASELINE_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy baseline best.pt: {BASELINE_BEST_MODEL_PATH}')
baseline_model_for_complexity = YOLO(str(BASELINE_BEST_MODEL_PATH))
baseline_ms, baseline_fps = benchmark_inference(baseline_model_for_complexity, speed_images)
multiseam_ms, multiseam_fps = benchmark_inference(best_model, speed_images)

complexity = pd.DataFrame([
    {
        'Model': 'YOLO26s',
        'Parameters': sum(parameter.numel() for parameter in baseline_model_for_complexity.model.parameters()),
        'GFLOPs': model_gflops(baseline_model_for_complexity),
        'Model size (MB)': BASELINE_BEST_MODEL_PATH.stat().st_size / (1024 ** 2),
        'Inference (ms/image)': baseline_ms,
        'FPS': baseline_fps,
    },
    {
        'Model': 'YOLO26s+MultiSEAM',
        'Parameters': sum(parameter.numel() for parameter in best_model.model.parameters()),
        'GFLOPs': model_gflops(best_model),
        'Model size (MB)': BEST_MODEL_PATH.stat().st_size / (1024 ** 2),
        'Inference (ms/image)': multiseam_ms,
        'FPS': multiseam_fps,
    },
])
display(complexity)

## 10. So sánh trực tiếp với YOLO26s baseline

Hai checkpoint được đánh giá trên cùng `data.yaml`, cùng `split='val'`, `imgsz=640` và `device=0`.

In [ ]:
baseline_comparison_metrics = baseline_model_for_complexity.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=False)
baseline_summary = extract_metrics(baseline_comparison_metrics, 'YOLO26s baseline')
multiseam_summary = validation_summary

def comparison_values(summary):
    no_helmet = summary['by_class']['rider_without_helmet']
    return {
        'Precision': summary['Precision'],
        'Recall': summary['Recall'],
        'F1': summary['F1'],
        'mAP50': summary['mAP50'],
        'mAP50-95': summary['mAP50-95'],
        'NoHelmet Precision': no_helmet['Precision'],
        'NoHelmet Recall': no_helmet['Recall'],
        'NoHelmet F1': no_helmet['F1'],
        'NoHelmet AP50': no_helmet['AP50'],
        'NoHelmet AP50-95': no_helmet['AP50-95'],
    }

baseline_values = comparison_values(baseline_summary)
multiseam_values = comparison_values(multiseam_summary)
comparison = pd.DataFrame({
    'Metric': list(baseline_values.keys()),
    'YOLO26s': list(baseline_values.values()),
    'YOLO26s+MultiSEAM': [multiseam_values[key] for key in baseline_values],
})
comparison['Delta'] = comparison['YOLO26s+MultiSEAM'] - comparison['YOLO26s']
display(comparison)

## 11. Quyết định E3 theo kết quả thực tế

Không hard-code baseline hoặc E3. MultiSEAM được giữ lại nếu ít nhất một mục tiêu chính (mAP50-95, AP50 hoặc Recall của `rider_without_helmet`) tăng và không làm mAP50-95 cùng AP50/Recall no-helmet giảm đồng thời.

In [ ]:
priority_metrics = ['mAP50-95', 'NoHelmet AP50', 'NoHelmet Recall']
priority_improvements = {metric: multiseam_values[metric] > baseline_values[metric] for metric in priority_metrics}
any_priority_improvement = any(priority_improvements.values())
joint_priority_drop = all(multiseam_values[metric] < baseline_values[metric] for metric in priority_metrics)
e3_decision = 'KEEP MultiSEAM' if any_priority_improvement and not joint_priority_drop else 'REJECT MultiSEAM'
print(f'Priority improvements: {priority_improvements}')
print(f'E3 decision: {e3_decision}')

## 12. Final audit trước khi lưu notebook

In [ ]:
assert MODEL_NAME == 'yolo26s.pt'
assert ultralytics.__version__ == ULTRALYTICS_VERSION == '8.4.155'
assert MULTISEAM_EXPECTED_COUNT == 2
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert TRAIN_CONFIG == {
    'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15, 'pretrained': True,
}
assert VALIDATION_SPLIT == 'val'
assert count_multiseam(custom_pytorch_model) == MULTISEAM_EXPECTED_COUNT == 2
assert trainer_multiseam_count == MULTISEAM_EXPECTED_COUNT == 2
assert multiseam_count_best == MULTISEAM_EXPECTED_COUNT == 2
assert trainer.model is custom_pytorch_model
assert test_metrics is None and test_summary is None
assert EXPERIMENT_NAME == 'helmet_yolo26s_multiseam_dataset2'
assert BASELINE_EXPERIMENT_NAME not in str(BEST_MODEL_PATH)
assert BEST_MODEL_PATH == OUTPUT_DIR / 'helmet_yolo26s_multiseam_dataset2' / 'weights' / 'best.pt'
assert DATA_YAML_PATH == Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
assert sum(1 for module in best_model.model.model if module.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}) == DETECT_HEAD_COUNT
assert EXPERIMENT_VERSION_FILE.exists()
assert ULTRALYTICS_VERSION in EXPERIMENT_VERSION_FILE.read_text(encoding='utf-8')
assert set(spec['stride'] for spec in fusion_specs).issubset({8, 16})
assert {spec['stride'] for spec in fusion_specs} == {8, 16}
assert set(comparison['Metric']) == set(baseline_values)

forbidden_modules = {'P2', 'iAFF', 'SEAM', 'DySample', 'SCConv', 'CBAM', 'ECA', 'AFPN', 'ADown'}
custom_module_names = {submodule.__class__.__name__ for submodule in best_model.model.modules()}
assert not (custom_module_names & forbidden_modules), custom_module_names & forbidden_modules

print('Final audit passed.')
print(f'Ultralytics version: {ultralytics.__version__}')
print(f'MultiSEAM modules before/train/best: {multiseam_before_train}/{trainer_multiseam_count}/{multiseam_count_best}')
print(f'Runtime fusion points: {[(spec["index"], spec["stride"]) for spec in fusion_specs]}')
print(f'Validation only: split={VALIDATION_SPLIT!r}; test disabled')
print(f'E3 checkpoint: {BEST_MODEL_PATH}')
print(f'Baseline untouched: {BASELINE_BEST_MODEL_PATH}')
print(f'Decision from measured metrics: {e3_decision}')